# Canonical lesion overlap — experimental groups × time points

This notebook visualizes **overlap of manually annotated lesion masks in Allen canonical space**: absolute mouse-count maps by group and time point, analytical ROI outlines, available-cortex windows, and a separate schematic of a representative mean-sized lesion.

- Time points: **PT (within 1.5 h) → 1-5h-PT → 1d-PT → 3d-PT**. `Day-PT` is baseline and **not** a lesion-assessment time point.
- Group membership is loaded **only** from the current YAML `analysis_selection.experimental_groups`. `analysis_exclusions` rules are applied **when generating these figures**, not in the annotation GUI.
- Records and `ref_stem` come from the **current** `out/step2_masks/metadata_registration.csv`, not from obsolete `damage_index.csv` or old `baseline_stem` values.
- Original `lesion_final.npy` files are **read only**. Spatial projection, absolute mouse-overlap counting, and the representative mean-lesion procedure follow the existing notebook implementation.
- The current `step6_atlas_transformation.py` and `step6b_fc_geometry_coverage.py` modules must be available from `src`, along with precomputed Step 6 transforms. This notebook does not recompute those atlas transforms.

**Quality control:** projection errors and absent masks are recorded in CSV files. Missing masks or transforms are **not** interpreted as zero lesion area. The output directory is `out/damage_assessment/canonical_lesion_overlap/` (without a version suffix).

The canonical figures mark bregma at **(X=228, Y=218)** in `atlas_data.flat_label` pixel coordinates (red dot with a white outline). This is a display-only marker and does not change masks or quantitative results.

**Execution note:** unlike the summary notebook's disabled primary export, this notebook has several saving flags enabled and **writes figures, CSV files, and NPY arrays** to its output directory when executed.


In [ ]:
%matplotlib inline

from __future__ import annotations

import importlib.util
import json
import sys
import warnings
from collections import OrderedDict
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from matplotlib import colors as mcolors
from matplotlib.colors import BoundaryNorm, ListedColormap
from skimage.segmentation import find_boundaries
from IPython.display import display


## 1. User settings


In [ ]:
# ========================= USER SETTINGS =========================

CONFIG_PATH = r"config.yaml"
SRC_DIR = r"...\src"

STEP6_PY_PATH = None
STEP6B_PY_PATH = None

EXP_DAYS = ["PT", "1-5h-PT", "1d-PT", "3d-PT"]
# None -> display every experimental group in YAML order; optional list reorders without exclusion.
GROUP_DISPLAY_ORDER = None


# -------- Language / display labels --------
LABEL_LANGUAGE = "en"   # "ru" or "en"

GROUP_LABELS_RU = {
    "Control": "Контроль",
    "Awake_PT": "ФТ_бодр",
    "Iso_PT": "ФТ_изо",
}
GROUP_LABELS_EN = {
    "Control": "Laser",
    "Awake_PT": "Awake PT",
    "Iso_PT": "IsO$_2$ PT",
}

EXP_DAY_LABELS_RU = {
    "PT": "PT (<1,5 ч)",
    "1-5h-PT": "1–5 ч",
    "1d-PT": "1 сутки",
    "3d-PT": "3 сутки",
}
EXP_DAY_LABELS_EN = {
    "PT": "10 min",
    "1-5h-PT": "1–5 h",
    "1d-PT": "1 day",
    "3d-PT": "3 days",
}

# -------- Bregma marker on canonical atlas images --------
SHOW_BREGMA = True
BREGMA_X = 228  # column in atlas_data.flat_label
BREGMA_Y = 218  # row in atlas_data.flat_label
BREGMA_COLOR = "#64161C"
BREGMA_SIZE = 50  # Matplotlib points squared
BREGMA_EDGE_COLOR = "white"
BREGMA_EDGE_WIDTH = 0.8

# -------- Canonical background --------
ATLAS_BACKGROUND_SCOPE = "cortex_only"   # "cortex_only" / "whole_brain"

NEUTRAL_ATLAS_STYLE = {
    "background": "#FFFFFF",
    "atlas_fill": "#F3F0E7",
    "fine_boundary": "#140700",
    "fine_boundary_alpha": 0.5,
    "fine_boundary_lw": 0.8,
    "outer_boundary": "#4F4B45",
    "outer_boundary_lw": 1.0,
}

# -------- Group-specific available cortex window --------
SHOW_GROUP_WINDOW = True
WINDOW_MASK_SET = None
# If None -> auto-select first available mask_set from Step6b canonical masks.

WINDOW_DIM_OUTSIDE = True
WINDOW_OUTSIDE_COLOR = "#7F847F"
WINDOW_OUTSIDE_ALPHA = 0.24

WINDOW_DRAW_FILL = False
WINDOW_FILL_COLOR = "#A9BDB4"
WINDOW_FILL_ALPHA = 0.07

WINDOW_DRAW_CONTOUR = True
WINDOW_CONTOUR_COLOR = "#27423D"
WINDOW_CONTOUR_LW = 2.0
WINDOW_CONTOUR_ALPHA = 0.95

# -------- Analytical ROI boundaries (SS, MOp, MOs, RSP) --------
SHOW_ANALYSIS_ROI_BOUNDARIES = True
ANALYSIS_ROIS_TO_SHOW = ["SS", "MOp", "MOs", "RSP"]

ROI_PALETTE = {
    "SS":  "#45766A",
    "MOp": "#A95843",
    "MOs": "#B78D36",
    "RSP": "#665078",
}

ANALYSIS_BOUNDARY_LINEWIDTH = 2
ANALYSIS_BOUNDARY_ALPHA = 0.5
ANALYSIS_BOUNDARY_DARKEN = 0.72

# Optional: also fill analytical ROIs very lightly underneath.
SHOW_ANALYSIS_ROI_FILL = False
ANALYSIS_FILL_ALPHA = 0.08

# -------- Count overlay (absolute mouse counts) --------
COUNT_CMAP = "YlOrRd"
COUNT_ALPHA = 0.82
# Manual overrides for low counts so that count=1 does not blend with the neutral atlas background.
COUNT_COLORS = {
    1: "#FDB366",
    2: "#F67E4B",
    3: "#E34A33",
    4: "#C61F1A",
    5: "#A00012",
    6: "#760019",
    7: "#480015",
}
COUNT_VMAX_OVERRIDE = None

SHOW_COUNT_BOUNDARY = False
COUNT_BOUNDARY_COLOR = "#4A0C00"
COUNT_BOUNDARY_LW = 0.8

# -------- Representative mean lesion scheme --------
SHOW_MEAN_LESION_SCHEME = True

# Union of all group windows as soft cohort window in the mean-lesion panel.
SHOW_MEAN_SCHEME_WINDOW = True

MEAN_LESION_FILL_COLOR = "#C65C3B"
MEAN_LESION_FILL_ALPHA = 0.34
MEAN_LESION_OUTLINE_COLOR = "#7F1D13"
MEAN_LESION_OUTLINE_LW = 2.6
MEAN_LESION_OUTLINE_ALPHA = 0.98

MEAN_SCHEME_FIGSIZE = (5, 5.0)
MEAN_SCHEME_TITLE = None  # e.g. "Mean lesion"

# -------- Figure layout --------
FIGSIZE_PER_PANEL = (2.4, 2)

GRID_WSPACE = 0.03
GRID_HSPACE = 0.08

MARGIN_LEFT = 0.05
MARGIN_RIGHT = 0.88
MARGIN_TOP = 0.90
MARGIN_BOTTOM = 0.06

COLORBAR_RECT = [0.90, 0.20, 0.018, 0.55]

TITLE_FONT_SIZE = 25
COLUMN_LABEL_FONT_SIZE = 25
ROW_LABEL_FONT_SIZE = 25
COLORBAR_LABEL_FONT_SIZE = 20
COLORBAR_TICK_FONT_SIZE = 20
N_TEXT_FONT_SIZE = 20
MEAN_SCHEME_TEXT_FONT_SIZE = 5

ROW_LABEL_X_FIG = 0.035
ROW_LABEL_ROTATION = 90

SUPTITLE = None
SUPTITLE_FONT_SIZE = 15

DPI = 240
SAVE_PNG = True
SAVE_TIFF = True
SAVE_SEPARATE_PANELS = True

INCLUDE_LESION_IN_PLOT_EXTENT = True
INCLUDE_GROUP_WINDOW_IN_PLOT_EXTENT = True
PLOT_PAD_PX = 4

OUTPUT_FOLDER_NAME = "canonical_lesion_overlap"

SAVE_PROJECTED_STEM_MASKS = True
SAVE_MOUSE_DAY_UNIONS = True
REUSE_EXISTING_PROJECTED_STEM_MASKS = False

FAIL_ON_MISSING_STEP6_TRANSFORM = False

# ================================================================

## 2. Imports from project source

`SRC_DIR` is an existing absolute local path defined in the settings cell. Update it for your installation or use the existing optional module-path settings. The import code has been preserved without modification.


In [ ]:
def _add_sys_path(path_like):
    """Add an existing directory or file parent to the Python module search path."""
    if path_like is None:
        return
    p = Path(path_like)
    if p.is_file():
        p = p.parent
    if p.exists():
        resolved = str(p.resolve())
        if resolved not in sys.path:
            sys.path.insert(0, resolved)

for candidate in [SRC_DIR, STEP6_PY_PATH, STEP6B_PY_PATH, Path.cwd(), Path.cwd() / "src", Path.cwd().parent / "src"]:
    _add_sys_path(candidate)


def _import_current_module(module_name, explicit_path=None):
    """Require the current module with a version-free name; no legacy-module fallbacks."""
    if explicit_path is not None:
        p = Path(explicit_path)
        if not p.is_file() or p.name != f"{module_name}.py":
            raise FileNotFoundError(f"Expected {module_name}.py, got: {p}")
        spec = importlib.util.spec_from_file_location(module_name, str(p))
        if spec is None or spec.loader is None:
            raise ImportError(f"Unable to load {module_name} from {p}")
        module = importlib.util.module_from_spec(spec)
        sys.modules[module_name] = module
        spec.loader.exec_module(module)
        return module
    try:
        return importlib.import_module(module_name)
    except ModuleNotFoundError as exc:
        if exc.name != module_name:
            raise
        raise ModuleNotFoundError(
            f"Не найден {module_name}.py без суффикса версии. "
            f"Проверьте SRC_DIR={SRC_DIR!r} и текущие файлы Step 6/6b."
        ) from exc

import importlib
s6 = _import_current_module("step6_atlas_transformation", STEP6_PY_PATH)
s6b = _import_current_module("step6b_fc_geometry_coverage", STEP6B_PY_PATH)
print("Step6  :", Path(s6.__file__))
print("Step6b :", Path(s6b.__file__))

## 3. Current paths, Step2 metadata, and canonical Allen atlas

`lesion_final.npy` files are read only from the **current** project directory. Step 6 supplies previously saved atlas geometry; Step2 metadata provides the complete recording list, including the new `PT` time point even if it is missing from the older Step 6 configuration's `analysis_selection.exp_days`.


In [ ]:
# Step6 supplies canonical Allen atlas and the existing Step6 transformations.
# Step2 remains authoritative for all stems (including PT not in older Step6 index filters).
s6_index_df, config, paths = s6.build_step6_index(
    CONFIG_PATH,
    include_bad=True,
    filters={},
    groups=None,
    stems=None,
    require_step3_annotation=False,
    require_response_roi=False,
)

# Always use the project resolved by the current Step6 API, not any old damage_index.csv path.
step2_cfg = config.get("step2", {}) or {}
metadata_csv = Path(paths["out_root"]) / str(step2_cfg.get("folder_name", "step2_masks")) / str(
    step2_cfg.get("metadata_registration_name", "metadata_registration.csv")
)
if not metadata_csv.is_file():
    raise FileNotFoundError(f"Actual Step2 registration metadata missing: {metadata_csv}")
registration = pd.read_csv(metadata_csv, low_memory=False)
required_meta = {"stem", "group_id", "ref_stem", "exp_day"}
missing_meta = required_meta - set(registration.columns)
if missing_meta:
    raise KeyError(f"Step2 metadata lacks {sorted(missing_meta)}: {metadata_csv}")
registration = registration.copy()
if registration["stem"].isna().any() or registration["stem"].astype(str).str.strip().eq("").any():
    raise ValueError("Step2 metadata has empty stem(s)")
if registration["stem"].astype(str).duplicated().any():
    raise ValueError("Step2 metadata has duplicate stem(s)")
for col in ("stem", "group_id", "ref_stem", "exp_day"):
    registration[col] = registration[col].astype(str)
if "mouse_id" not in registration.columns:
    registration["mouse_id"] = registration["stem"].str.split("_", n=1).str[0]
else:
    missing_mouse = registration["mouse_id"].isna() | registration["mouse_id"].astype(str).str.strip().eq("")
    registration.loc[missing_mouse, "mouse_id"] = registration.loc[missing_mouse, "stem"].str.split("_", n=1).str[0]
registration["mouse_id"] = registration["mouse_id"].astype(str)

# Enrich Step2 rows only with additional Step6 columns; never override current Step2
# registration values or ref_stem with potentially older Step6 index values.
index_df = registration.copy()
if not s6_index_df.empty and {"stem", "group_id"}.issubset(s6_index_df.columns):
    extras = [c for c in s6_index_df.columns if c not in index_df.columns]
    if extras:
        from_step6 = s6_index_df[["stem", "group_id"] + extras].copy()
        from_step6["stem"] = from_step6["stem"].astype(str)
        from_step6["group_id"] = from_step6["group_id"].astype(str)
        if from_step6.duplicated(["stem", "group_id"]).any():
            raise ValueError("Step6 index has duplicate stem/group_id pairs")
        index_df = index_df.merge(from_step6, on=["stem", "group_id"], how="left", validate="one_to_one")

s6b_paths = s6b.build_step6b_paths(config, paths)
atlas_data = s6.load_allen_atlas(config)
canonical_shape = tuple(atlas_data.flat_label.shape)

damage_cfg = config.get("damage_assessment", {}) or {}
damage_out_cfg = damage_cfg.get("output", {}) or {}
damage_root = Path(paths["out_root"]) / str(damage_cfg.get("folder_name", "damage_assessment"))
damage_masks_root = damage_root / str(damage_out_cfg.get("masks_folder", "masks"))
if not damage_masks_root.is_dir():
    raise FileNotFoundError(f"Lesion masks folder missing: {damage_masks_root}")

output_root = damage_root / OUTPUT_FOLDER_NAME
projected_stem_root = output_root / "projected_stem_masks"
mouse_day_root = output_root / "mouse_day_union_masks"
count_maps_root = output_root / "count_maps"
tables_root = output_root / "tables"
figures_root = output_root / "figures"
for folder in [output_root, projected_stem_root, mouse_day_root, count_maps_root, tables_root, figures_root]:
    folder.mkdir(parents=True, exist_ok=True)

if len(set(EXP_DAYS)) != len(EXP_DAYS) or "PT" not in EXP_DAYS or "Day-PT" in EXP_DAYS:
    raise ValueError("EXP_DAYS must include PT once and exclude baseline Day-PT")
print("Registration metadata:", metadata_csv)
print("Step2 rows / Step6 indexed rows:", len(registration), "/", len(s6_index_df))
print("Canonical shape:", canonical_shape)
print("Lesion masks root:", damage_masks_root)
print("Output root:", output_root)

## 4. Experimental groups and exclusions from the current YAML

Group membership is taken from `analysis_selection.experimental_groups`; no manually curated mouse list is used. `analysis_exclusions.rules` use **exact metadata-field matches** (AND between fields, OR among values in a list). They are evaluated against complete Step2 metadata, including baseline `Day-PT`, so `fail_on_unmatched_rules` is meaningful even when there is no baseline lesion mask. Separate selection and exclusion-rule audit CSV files are produced.


In [ ]:
def _mouse_list(value):
    """Normalize YAML mouse membership values into a list of ID strings."""
    if isinstance(value, dict):
        for key in ("mice", "mouse_ids", "include_mice", "members"):
            if key in value:
                return _mouse_list(value[key])
        raise ValueError(f"Group specification has no mice: {value!r}")
    if value is None:
        return []
    if isinstance(value, str):
        return [value.strip()]
    if isinstance(value, (list, tuple, set)):
        return [str(x).strip() for x in value]
    raise TypeError(f"Unexpected mouse list: {value!r}")

selection_cfg = config.get("analysis_selection", {}) or {}
raw_groups = selection_cfg.get("experimental_groups")
if not isinstance(raw_groups, dict) or not raw_groups:
    raise ValueError("Current YAML must have analysis_selection.experimental_groups")
analysis_groups = OrderedDict((str(name), _mouse_list(mice)) for name, mice in raw_groups.items())
mouse_to_analysis_group = {}
for group_name, mice in analysis_groups.items():
    for mouse in mice:
        if mouse in mouse_to_analysis_group:
            raise ValueError(f"Mouse {mouse} belongs to two groups: {mouse_to_analysis_group[mouse]}, {group_name}")
        mouse_to_analysis_group[mouse] = group_name
if GROUP_DISPLAY_ORDER is None:
    display_order = list(analysis_groups)
else:
    unknown = set(GROUP_DISPLAY_ORDER) - set(analysis_groups)
    if unknown:
        raise ValueError(f"GROUP_DISPLAY_ORDER has names absent from YAML: {sorted(unknown)}")
    display_order = list(dict.fromkeys([*GROUP_DISPLAY_ORDER, *analysis_groups.keys()]))
include_mice_cfg = selection_cfg.get("include_mice")
include_mice = set(_mouse_list(include_mice_cfg)) if include_mice_cfg is not None else None
require_group_assignment = bool(selection_cfg.get("require_group_assignment", False))

# Apply exactly the configured exclusion rules against FULL Step2 metadata.
# In particular, a Day-PT-only rule must not exclude a poststroke day of the same mouse.
exclusions_cfg = config.get("analysis_exclusions", {}) or {}
apply_exclusions = bool(exclusions_cfg.get("enabled", False))
exclusion_rules = exclusions_cfg.get("rules", []) or []
if not isinstance(exclusion_rules, list):
    raise TypeError("analysis_exclusions.rules must be a list")

def _match_exclusion_rule(df, conditions):
    """Match a rule against metadata using AND across fields and OR within lists."""
    if not isinstance(conditions, dict) or not conditions:
        raise ValueError("Exclusion match must be a nonempty mapping")
    matched = pd.Series(True, index=df.index)
    for field, allowed in conditions.items():
        if field not in df.columns:
            raise KeyError(f"Exclusion rule field absent in Step2 metadata: {field!r}")
        variants = allowed if isinstance(allowed, (list, tuple, set)) else [allowed]
        if not variants:
            raise ValueError(f"Empty exclusion match list for {field}")
        matched &= df[field].notna() & df[field].astype(str).isin([str(v) for v in variants])
    return matched

rule_audit_rows = []
exclusion_by_stem = {}
seen_rule_ids = set()
if apply_exclusions:
    for index, rule in enumerate(exclusion_rules, start=1):
        if not isinstance(rule, dict):
            raise TypeError(f"Exclusion rule #{index} must be a mapping")
        rid = str(rule.get("id") or f"rule_{index}")
        if rid in seen_rule_ids:
            raise ValueError(f"Duplicate exclusion rule ID: {rid}")
        seen_rule_ids.add(rid)
        hit = _match_exclusion_rule(registration, rule.get("match"))
        stems = registration.loc[hit, "stem"].astype(str).tolist()
        for stem in stems:
            exclusion_by_stem.setdefault(stem, []).append((rid, str(rule.get("reason", ""))))
        rule_audit_rows.append({
            "rule_id": rid,
            "reason": str(rule.get("reason", "")),
            "match_json": json.dumps(rule["match"], ensure_ascii=False, sort_keys=True),
            "matched_metadata_stems": len(stems),
            "matched_target_day_stems": int((registration.loc[hit, "exp_day"].isin(EXP_DAYS)).sum()),
            "status": "matched" if stems else "unmatched",
        })

rule_audit = pd.DataFrame(rule_audit_rows, columns=[
    "rule_id", "reason", "match_json", "matched_metadata_stems", "matched_target_day_stems", "status"
])
if apply_exclusions and not rule_audit.empty and rule_audit["status"].eq("unmatched").any():
    ids = rule_audit.loc[rule_audit["status"].eq("unmatched"), "rule_id"].tolist()
    message = f"Exclusion rule(s) unmatched in FULL Step2 metadata: {ids}"
    if bool(exclusions_cfg.get("fail_on_unmatched_rules", False)):
        raise RuntimeError(message)
    warnings.warn(message)

index_df = index_df.copy()
index_df["analysis_group"] = index_df["mouse_id"].map(mouse_to_analysis_group)
index_df["lesion_final_path"] = index_df["stem"].map(lambda stem: str(damage_masks_root / str(stem) / "lesion_final.npy"))
index_df["lesion_exists"] = index_df["lesion_final_path"].map(lambda p: Path(p).is_file())
index_df["exclusion_rule_ids"] = index_df["stem"].map(
    lambda s: ";".join(x[0] for x in exclusion_by_stem.get(str(s), []))
)
index_df["exclusion_reasons"] = index_df["stem"].map(
    lambda s: ";".join(x[1] for x in exclusion_by_stem.get(str(s), []))
)

index_df["selection_status"] = "pending"
index_df.loc[~index_df["exp_day"].isin(EXP_DAYS), "selection_status"] = "outside_target_days"
if include_mice is not None:
    mask = index_df["selection_status"].eq("pending") & ~index_df["mouse_id"].isin(include_mice)
    index_df.loc[mask, "selection_status"] = "outside_analysis_include_mice"
mask = index_df["selection_status"].eq("pending") & index_df["exclusion_rule_ids"].ne("")
index_df.loc[mask, "selection_status"] = "excluded_by_analysis_exclusions"
mask = index_df["selection_status"].eq("pending") & index_df["analysis_group"].isna()
index_df.loc[mask, "selection_status"] = "mouse_not_in_experimental_groups"
if require_group_assignment and mask.any():
    raise ValueError("Missing experimental group for: " + ", ".join(sorted(index_df.loc[mask, "mouse_id"].unique())))
mask = index_df["selection_status"].eq("pending") & ~index_df["lesion_exists"]
index_df.loc[mask, "selection_status"] = "missing_lesion_mask"
index_df.loc[index_df["selection_status"].eq("pending"), "selection_status"] = "included"

selection_audit = index_df[[c for c in [
    "analysis_group", "mouse_id", "group_id", "stem", "ref_stem", "exp_day", "state", "protocol",
    "lesion_final_path", "lesion_exists", "exclusion_rule_ids", "exclusion_reasons", "selection_status"
] if c in index_df.columns]].copy()
selection_audit.to_csv(tables_root / "lesion_selection_audit.csv", index=False)
rule_audit.to_csv(tables_root / "analysis_exclusion_rule_audit.csv", index=False)

selected = index_df[index_df["selection_status"].eq("included")].copy()
missing_masks = index_df[index_df["selection_status"].eq("missing_lesion_mask")].copy()
print("Groups (from YAML):")
for group_name in display_order:
    print(f"  {group_name}: {', '.join(analysis_groups[group_name])}")
print("Exclusions enabled:", apply_exclusions)
print("Selected lesion stems:", len(selected))
print("Target-day missing lesion masks:", len(missing_masks))
display(selection_audit["selection_status"].value_counts().rename_axis("status").reset_index(name="n_stems"))
display(rule_audit)
if selected.empty:
    raise RuntimeError("No lesion masks selected. See lesion_selection_audit.csv and Step2 metadata.")

## 5. Plotting helpers, canonical anatomical background, and analytical ROI masks

Display styles and ROI boundaries are used for visualization. The atlas and analysis ROI arrays are not altered by drawing functions.


In [ ]:
def localized_group_label(group_name):
    """Return the selected English or Russian label for an experimental group."""
    d = GROUP_LABELS_RU if str(LABEL_LANGUAGE).lower() == "ru" else GROUP_LABELS_EN
    return d.get(str(group_name), str(group_name))


def localized_exp_day_label(day):
    """Return the selected display label for an experimental time point."""
    d = EXP_DAY_LABELS_RU if str(LABEL_LANGUAGE).lower() == "ru" else EXP_DAY_LABELS_EN
    return d.get(str(day), str(day))


def cortex_mask_from_flat_label(atlas_data):
    """Derive the cortical mask from the flat Allen atlas label image."""
    ids = s6.find_structure_ids(
        atlas_data,
        ["Isocortex"],
        include_descendants=True,
    )
    return np.isin(
        np.asarray(atlas_data.flat_label, dtype=np.int32),
        np.asarray(sorted(ids), dtype=np.int32),
    )


def background_from_scope(atlas_data, scope):
    """Select the atlas background mask for the configured anatomical scope."""
    labels = np.asarray(atlas_data.flat_label, dtype=np.int32)
    scope = str(scope).lower()
    if scope == "whole_brain":
        mask = labels > 0
        labelmap = labels.copy()
    elif scope == "cortex_only":
        mask = cortex_mask_from_flat_label(atlas_data)
        labelmap = labels.copy()
        labelmap[~mask] = 0
    else:
        raise ValueError(f"Unknown ATLAS_BACKGROUND_SCOPE={scope!r}")
    return labelmap, mask


background_labelmap, display_mask = background_from_scope(atlas_data, ATLAS_BACKGROUND_SCOPE)
detailed_boundaries = find_boundaries(background_labelmap, mode="inner")

if SHOW_BREGMA:
    atlas_h, atlas_w = canonical_shape
    if not (0 <= BREGMA_X < atlas_w and 0 <= BREGMA_Y < atlas_h):
        raise ValueError(
            f"Bregma (x={BREGMA_X}, y={BREGMA_Y}) outside canonical atlas "
            f"(width={atlas_w}, height={atlas_h})"
        )


def draw_fill(ax, mask, color, alpha=0.25, zorder=5):
    """Draw a transparent mask fill on Matplotlib axes for visualization."""
    m = np.asarray(mask, dtype=bool)
    if not np.any(m):
        return
    overlay = np.zeros((m.shape[0], m.shape[1], 4), dtype=float)
    rgb = mcolors.to_rgb(color)
    overlay[m, :3] = rgb
    overlay[m, 3] = float(alpha)
    ax.imshow(overlay, interpolation="nearest", zorder=zorder)


def draw_boundary(ax, boundary_mask, color="black", lw=1.0, alpha=1.0, zorder=10):
    """Draw a binary boundary overlay without changing underlying data."""
    b = np.asarray(boundary_mask, dtype=bool)
    if np.any(b):
        ax.contour(
            b.astype(float),
            levels=[0.5],
            colors=[color],
            linewidths=lw,
            alpha=alpha,
            zorder=zorder,
        )


def darken_hex(color, factor=0.72):
    """Scale an RGB hexadecimal display color by a darkening factor."""
    rgb = np.array(mcolors.to_rgb(color))
    return mcolors.to_hex(np.clip(rgb * float(factor), 0, 1))


def neutral_atlas(ax):
    """Render a neutral atlas background and cortical/region boundaries."""
    cfg = NEUTRAL_ATLAS_STYLE
    ax.set_facecolor(cfg["background"])
    draw_fill(ax, display_mask, cfg["atlas_fill"], alpha=1.0, zorder=1)
    draw_boundary(
        ax,
        detailed_boundaries,
        cfg["fine_boundary"],
        lw=cfg["fine_boundary_lw"],
        alpha=cfg["fine_boundary_alpha"],
        zorder=3,
    )
    outer = find_boundaries(np.asarray(display_mask, dtype=np.uint8), mode="outer")
    draw_boundary(
        ax,
        outer,
        cfg["outer_boundary"],
        lw=cfg["outer_boundary_lw"],
        alpha=0.90,
        zorder=4,
    )


def draw_group_window_overlay(ax, mask):
    """Display an available-cortex window as optional fill, dimming, and outline."""
    m = np.asarray(mask, dtype=bool)
    if not np.any(m):
        return

    if WINDOW_DIM_OUTSIDE:
        outside = np.asarray(display_mask, dtype=bool) & (~m)
        draw_fill(
            ax,
            outside,
            color=WINDOW_OUTSIDE_COLOR,
            alpha=WINDOW_OUTSIDE_ALPHA,
            zorder=5.2,
        )

    if WINDOW_DRAW_FILL:
        draw_fill(
            ax,
            m,
            color=WINDOW_FILL_COLOR,
            alpha=WINDOW_FILL_ALPHA,
            zorder=5.3,
        )

    if WINDOW_DRAW_CONTOUR:
        boundary = find_boundaries(m.astype(np.uint8), mode="outer")
        draw_boundary(
            ax,
            boundary,
            color=WINDOW_CONTOUR_COLOR,
            lw=WINDOW_CONTOUR_LW,
            alpha=WINDOW_CONTOUR_ALPHA,
            zorder=12,
        )


def crop_from_mask(mask, pad=3):
    """Determine a padded XY plotting extent from a binary canonical mask.
    
    Returns
    -------
    tuple of float
        ``(xmin, xmax, ymax, ymin)`` for image-style display.
    """
    m = np.asarray(mask, dtype=bool)
    yy, xx = np.where(m)
    if yy.size == 0:
        return (-0.5, m.shape[1] - 0.5, m.shape[0] - 0.5, -0.5)
    x0 = max(int(xx.min()) - int(pad), 0)
    x1 = min(int(xx.max()) + int(pad), m.shape[1] - 1)
    y0 = max(int(yy.min()) - int(pad), 0)
    y1 = min(int(yy.max()) + int(pad), m.shape[0] - 1)
    return (x0 - 0.5, x1 + 0.5, y1 + 0.5, y0 - 0.5)


def style_axes(ax, extent):
    """Apply equal-aspect image limits and hide Matplotlib axis decorations."""
    ax.set_xlim(extent[0], extent[1])
    ax.set_ylim(extent[2], extent[3])
    ax.set_aspect("equal")
    ax.axis("off")


def draw_bregma_marker(ax):
    """Display-only marker in full canonical pixel coordinates (x=column, y=row)."""
    if not SHOW_BREGMA:
        return
    ax.scatter(
        [BREGMA_X],
        [BREGMA_Y],
        s=BREGMA_SIZE,
        c=BREGMA_COLOR,
        edgecolors=BREGMA_EDGE_COLOR,
        linewidths=BREGMA_EDGE_WIDTH,
        zorder=100,
        clip_on=True,
    )


# ---- Analytical Step6b ROI map (same canonical space) ----
def canonical_analysis_from_existing(s6b_paths):
    """Load the canonical analytical ROI image and definitions from Step6b outputs."""
    root = Path(s6b_paths["canonical_root"])
    preferred = root / "analysis_roi_canonical.npy"
    roi_csv = Path(s6b_paths["direct_children_csv"])
    if preferred.exists() and roi_csv.exists():
        arr = np.load(preferred)
        defs = pd.read_csv(roi_csv)
        return arr.astype(np.int32), defs, "loaded from Step6b outputs"
    return None, None, None


def canonical_analysis_build_on_the_fly(atlas_data, config, paths):
    """Construct canonical broad-area ROI labels using existing Step6b functions."""
    parent_query = str(
        s6b.cfg_get(
            config,
            "step6b_fc_geometry",
            "broad_atlas",
            "parent_acronym",
            default="Isocortex",
        )
    )
    built = s6b.build_canonical_side_rois(atlas_data, config, paths, parent_query)
    arr = built["broad_full"].astype(np.int32)
    defs = built["children"].copy()
    return arr, defs, "built on the fly"


def prepare_analysis_map(atlas_data, config, paths, s6b_paths):
    """Load or construct an analytical ROI map and normalize its definition columns."""
    arr = defs = source = None
    arr, defs, source = canonical_analysis_from_existing(s6b_paths)
    if arr is None:
        arr, defs, source = canonical_analysis_build_on_the_fly(atlas_data, config, paths)

    defs = defs.copy()
    if "analysis_structure_id" not in defs.columns and "broad_structure_id" in defs.columns:
        defs["analysis_structure_id"] = defs["broad_structure_id"]
    if "analysis_roi_acronym" not in defs.columns and "broad_acronym" in defs.columns:
        defs["analysis_roi_acronym"] = defs["broad_acronym"]
    if "analysis_roi_name" not in defs.columns and "broad_name" in defs.columns:
        defs["analysis_roi_name"] = defs["broad_name"]

    defs = defs.drop_duplicates("analysis_structure_id").copy()
    return arr, defs, source


def make_analysis_masks(analysis_map, defs, selected_acronyms=None):
    """Build named binary ROI masks from a canonical integer label image."""
    rows = []
    if selected_acronyms is not None:
        wanted = {str(x) for x in selected_acronyms}
        defs = defs[defs["analysis_roi_acronym"].astype(str).isin(wanted)].copy()

    defs = defs.sort_values(["analysis_roi_acronym", "analysis_structure_id"])
    for _, r in defs.iterrows():
        aid = int(r["analysis_structure_id"])
        acr = str(r["analysis_roi_acronym"])
        name = str(r.get("analysis_roi_name", acr))
        mask = np.asarray(analysis_map == aid, dtype=bool)
        if np.any(mask):
            rows.append(
                {
                    "analysis_structure_id": aid,
                    "analysis_roi_acronym": acr,
                    "analysis_roi_name": name,
                    "mask": mask,
                    "color": ROI_PALETTE.get(acr, "#808080"),
                }
            )
    order = {a: i for i, a in enumerate(ANALYSIS_ROIS_TO_SHOW)}
    return sorted(rows, key=lambda x: order.get(x["analysis_roi_acronym"], 999))


analysis_map, analysis_defs, analysis_source = prepare_analysis_map(
    atlas_data, config, paths, s6b_paths
)
analysis_items = make_analysis_masks(
    analysis_map,
    analysis_defs,
    selected_acronyms=ANALYSIS_ROIS_TO_SHOW,
)
print("Analytical ROI source:", analysis_source)
display(
    pd.DataFrame([
        {
            "acronym": it["analysis_roi_acronym"],
            "name": it["analysis_roi_name"],
            "area_px": int(np.sum(it["mask"])),
        }
        for it in analysis_items
    ])
)


def draw_analysis_roi_boundaries(ax):
    """Overlay selected analytical ROI boundaries on the canonical figure."""
    if not SHOW_ANALYSIS_ROI_BOUNDARIES:
        return

    for it in analysis_items:
        mask = np.asarray(it["mask"], dtype=bool) & np.asarray(display_mask, dtype=bool)

        if SHOW_ANALYSIS_ROI_FILL:
            draw_fill(
                ax,
                mask,
                color=it["color"],
                alpha=ANALYSIS_FILL_ALPHA,
                zorder=5.6,
            )

        boundary = find_boundaries(mask.astype(np.uint8), mode="inner")
        draw_boundary(
            ax,
            boundary,
            color=darken_hex(it["color"], ANALYSIS_BOUNDARY_DARKEN),
            lw=ANALYSIS_BOUNDARY_LINEWIDTH,
            alpha=ANALYSIS_BOUNDARY_ALPHA,
            zorder=13,
        )


## 6. Step6b geometry and lesion projection

The existing Step 6 atlas transform and effective stem-to-reference registration are used to sample each native lesion mask in canonical space. Original native masks remain unchanged.


In [ ]:
manual_overrides = s6b.load_manual_registration_overrides(config, paths)
_projection_cache = {}


def _load_json(path):
    """Read a JSON file containing a saved atlas transformation."""
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def get_group_projection(row):
    """Cache canonical-to-reference projection geometry for a group/reference pair.
    
    Notes
    -----
    Uses the saved Step6 atlas transform. The reference image geometry
    is read from the current project's images when available.
    """
    gid = str(row["group_id"])
    ref_stem = str(row["ref_stem"])
    key = (gid, ref_stem)

    if key in _projection_cache:
        return _projection_cache[key]

    group_rows = index_df[index_df["group_id"].astype(str).eq(gid)].copy()
    ref_rows = group_rows[group_rows["stem"].astype(str).eq(ref_stem)]
    # Prefer the reference BMP in the CURRENT analysis project.
    # Do not read a stale saved F:/C: path if the historical folder still exists.
    import cv2
    ref_shape = None
    candidates = [Path(paths["reference_images_root"]) / "505" / f"{ref_stem}.bmp"]
    for candidate in candidates:
        if candidate.is_file():
            img = cv2.imread(str(candidate), cv2.IMREAD_UNCHANGED)
            if img is not None:
                ref_shape = tuple(img.shape[:2])
                break
    # Only if the current Step1 image is missing use the existing Step6 loader.
    if ref_shape is None and not ref_rows.empty:
        try:
            ref_img = s6.load_reference_image_for_row(ref_rows.iloc[-1])
            ref_shape = tuple(ref_img.shape[:2])
        except (FileNotFoundError, OSError, KeyError):
            pass
    if ref_shape is None:
        raise FileNotFoundError(f"Reference image unavailable for {gid}, ref_stem={ref_stem}")

    transform_path = s6b.group_transform_path(paths, gid, ref_stem)
    if not transform_path.exists():
        raise FileNotFoundError(f"Step6 atlas transform missing: {transform_path}")

    atlas_transform = _load_json(transform_path)
    projection = s6b.project_canonical_grid_to_reference(
        atlas_data,
        atlas_transform,
        ref_shape,
        config,
    )

    bundle = {
        "projection": projection,
        "ref_shape": ref_shape,
        "ref_stem": ref_stem,
        "group_id": gid,
        "atlas_transform_path": str(transform_path),
    }
    _projection_cache[key] = bundle
    return bundle


def project_lesion_stem_to_canonical(row):
    """Project a native binary lesion mask into the canonical Allen grid.
    
    Parameters
    ----------
    row : pandas.Series
        Selected Step2 metadata record with ``stem`` and lesion-mask path.
    
    Returns
    -------
    canonical_mask : numpy.ndarray of bool
        Binary mask in ``atlas_data.flat_label`` coordinates.
    metadata : dict
        Projection source, geometry, and area QC fields.
    
    Notes
    -----
    The source ``lesion_final.npy`` is read only. Cached canonical masks
    may be reused when the corresponding flag is enabled.
    """
    stem = str(row["stem"])
    lesion_path = Path(row["lesion_final_path"])
    cache_path = projected_stem_root / f"{stem}__lesion_canonical.npy"

    if REUSE_EXISTING_PROJECTED_STEM_MASKS and cache_path.exists():
        can = np.load(cache_path).astype(bool)
        return can, {
            "projection_source": "cached",
            "transform_source": "cached",
            "native_area_px": np.nan,
            "reference_area_px": np.nan,
            "canonical_area_px": int(can.sum()),
            "projected_mask_path": str(cache_path),
        }

    lesion_native = np.load(lesion_path).astype(bool)
    bundle = get_group_projection(row)

    tform, transform_source, tform_values = s6b.effective_tform_stem_to_ref(
        row, manual_overrides, config
    )
    lesion_ref = s6b.warp_bool_stem_to_ref(
        lesion_native,
        tform,
        bundle["ref_shape"],
    )
    lesion_can = s6b.sample_ref_mask_on_canonical(
        bundle["projection"],
        lesion_ref,
    ).astype(bool)

    if SAVE_PROJECTED_STEM_MASKS:
        np.save(cache_path, lesion_can.astype(np.uint8))

    return lesion_can, {
        "projection_source": "computed",
        "transform_source": transform_source,
        "translation_x": float(tform_values["translation_x"]),
        "translation_y": float(tform_values["translation_y"]),
        "rotation_rad": float(tform_values["rotation_rad"]),
        "native_area_px": int(lesion_native.sum()),
        "reference_area_px": int(lesion_ref.sum()),
        "canonical_area_px": int(lesion_can.sum()),
        "projected_mask_path": str(cache_path),
        "atlas_transform_path": str(bundle["atlas_transform_path"]),
    }

## 7. Project all lesion masks and union within `mouse × exp_day`

When more than one successful recording exists for a mouse at one time point, its canonical masks are combined by a **logical union** before counting mice. Failed projections are retained in QC outputs, not replaced by empty masks.


In [ ]:
projection_records = []
canonical_masks_by_stem = {}

for _, row in selected.iterrows():
    stem = str(row["stem"])
    base = {
        "analysis_group": str(row["analysis_group"]),
        "mouse_id": str(row["mouse_id"]),
        "group_id": str(row["group_id"]),
        "stem": stem,
        "ref_stem": str(row["ref_stem"]),
        "exp_day": str(row["exp_day"]),
        "state": str(row.get("state", "")),
        "protocol": str(row.get("protocol", "")),
        "lesion_final_path": str(row["lesion_final_path"]),
    }
    try:
        can_mask, meta = project_lesion_stem_to_canonical(row)
        canonical_masks_by_stem[stem] = can_mask
        projection_records.append({**base, **meta, "projection_ok": True, "error": ""})
    except Exception as exc:
        projection_records.append({**base, "projection_ok": False, "error": repr(exc)})
        if FAIL_ON_MISSING_STEP6_TRANSFORM:
            raise

projection_qc = pd.DataFrame(projection_records)
projection_qc.to_csv(tables_root / "lesion_canonical_projection_qc.csv", index=False)
display(projection_qc)

mouse_day_records = []
mouse_day_masks = {}

ok_qc = projection_qc[projection_qc["projection_ok"].astype(bool)].copy()

for (analysis_group, mouse_id, exp_day), g in ok_qc.groupby(
    ["analysis_group", "mouse_id", "exp_day"],
    sort=False,
):
    stems = g["stem"].astype(str).tolist()
    masks = [canonical_masks_by_stem[stem] for stem in stems]
    union_mask = np.logical_or.reduce(masks).astype(bool)

    key = (str(analysis_group), str(mouse_id), str(exp_day))
    mouse_day_masks[key] = union_mask

    out_path = mouse_day_root / f"{mouse_id}__{exp_day}__lesion_union_canonical.npy"
    if SAVE_MOUSE_DAY_UNIONS:
        np.save(out_path, union_mask.astype(np.uint8))

    mouse_day_records.append(
        {
            "analysis_group": str(analysis_group),
            "mouse_id": str(mouse_id),
            "exp_day": str(exp_day),
            "n_stems_used": len(stems),
            "stems_used": ";".join(stems),
            "canonical_union_area_px": int(union_mask.sum()),
            "union_mask_path": str(out_path),
        }
    )

mouse_day_qc = pd.DataFrame(mouse_day_records)
mouse_day_qc.to_csv(tables_root / "lesion_mouse_day_canonical_qc.csv", index=False)
display(mouse_day_qc)


## 8. Group-specific available-cortex windows

The `mask_set` name comes from the current `step6b_fc_geometry.primary_roi_mask_set`, rather than being inferred from a directory name. If a group has no available canonical technical masks, its window overlay is skipped with a warning.


In [ ]:
def discover_window_mask_set(config, canonical_masks_root: Path):
    """Select the Step6b technical mask set for available-cortex overlays."""
    fc_cfg = config.get("step6b_fc_geometry", {}) or {}
    # Prefer explicit, current Step6b mask-set names; names contain underscores.
    primary = fc_cfg.get("primary_roi_mask_set")
    if primary:
        return str(primary)
    mask_sets = fc_cfg.get("mask_sets", {}) or {}
    for name, spec in mask_sets.items():
        if not isinstance(spec, dict) or spec.get("enabled", True):
            return str(name)
    return None


canonical_masks_root = Path(s6b_paths["canonical_masks_root"])
window_mask_set = WINDOW_MASK_SET or discover_window_mask_set(config, canonical_masks_root)
print("Window mask set:", window_mask_set)

if SHOW_GROUP_WINDOW and window_mask_set is None:
    warnings.warn(
        "Could not auto-detect WINDOW_MASK_SET from Step6b canonical masks root. "
        "Group window overlay will be disabled."
    )
    SHOW_GROUP_WINDOW = False


def load_group_window_mask(group_name, group_rows):
    """Union available-cortex masks for the selected group's registration IDs.
    
    Returns
    -------
    mask : numpy.ndarray of bool or None
        Canonical group window, or None if no matching masks exist.
    loaded : list of dict
        Group IDs and source paths contributing to the union.
    """
    if not SHOW_GROUP_WINDOW:
        return None, []

    gid_values = (
        group_rows["group_id"].astype(str).drop_duplicates().tolist()
        if "group_id" in group_rows.columns
        else group_rows["mouse_id"].astype(str).drop_duplicates().tolist()
    )

    loaded = []
    union_mask = np.zeros(canonical_shape, dtype=bool)

    for gid in gid_values:
        try:
            folder_name = s6b.safe_key(window_mask_set, gid)
        except Exception:
            folder_name = f"{window_mask_set}__{gid}"

        p = canonical_masks_root / folder_name / "fixed_technical_canonical.npy"
        if not p.exists():
            continue

        arr = np.asarray(np.load(p), dtype=bool)
        if arr.shape != canonical_shape:
            warnings.warn(
                f"Window mask shape mismatch for {gid}: {arr.shape} != {canonical_shape}. Skip."
            )
            continue
        union_mask |= arr
        loaded.append({"group_id": gid, "path": str(p)})

    if not loaded:
        warnings.warn(
            f"No group window masks were loaded for analysis_group={group_name!r} "
            f"with mask_set={window_mask_set!r}."
        )
        return None, []

    return union_mask.astype(bool), loaded


group_window_masks = {}
group_window_rows = []

for group_name in display_order:
    rows = selected[selected["analysis_group"].astype(str).eq(str(group_name))].copy()
    mask, loaded = load_group_window_mask(group_name, rows)
    group_window_masks[str(group_name)] = mask
    group_window_rows.append(
        {
            "analysis_group": str(group_name),
            "mask_set": window_mask_set,
            "window_available": mask is not None,
            "n_loaded_mouse_masks": len(loaded),
            "window_area_px": int(mask.sum()) if mask is not None else 0,
            "loaded_group_ids": ";".join([r["group_id"] for r in loaded]),
        }
    )

group_window_summary = pd.DataFrame(group_window_rows)
group_window_summary.to_csv(tables_root / "group_available_window_summary.csv", index=False)
display(group_window_summary)

## 9. Mouse-count maps and discrete count colormap

Each canonical pixel is counted at most once per mouse and time point, after within-mouse union. Map values are **absolute mouse counts**, not percentages or fractions.


In [ ]:
count_maps = {}
group_day_records = []

for group_name in display_order:
    for day in EXP_DAYS:
        keys = [
            key
            for key in mouse_day_masks
            if key[0] == str(group_name) and key[2] == str(day)
        ]

        if keys:
            stack = np.stack(
                [mouse_day_masks[key].astype(np.uint16) for key in keys],
                axis=0,
            )
            count_map = stack.sum(axis=0).astype(np.uint16)
        else:
            count_map = np.zeros(canonical_shape, dtype=np.uint16)

        count_maps[(str(group_name), str(day))] = count_map

        map_path = count_maps_root / f"{group_name}__{day}__lesion_count_canonical.npy"
        np.save(map_path, count_map)

        group_day_records.append(
            {
                "analysis_group": str(group_name),
                "exp_day": str(day),
                "n_mice": len(keys),
                "max_count": int(count_map.max()),
                "n_nonzero_canonical_pixels": int((count_map > 0).sum()),
                "count_map_path": str(map_path),
                "mouse_ids": ";".join(sorted(set([key[1] for key in keys]))),
            }
        )

group_day_summary = pd.DataFrame(group_day_records)
group_day_summary.to_csv(tables_root / "lesion_group_day_canonical_summary.csv", index=False)
display(group_day_summary)

observed_global_max = max([int(arr.max()) for arr in count_maps.values()] + [0])
count_vmax = int(COUNT_VMAX_OVERRIDE) if COUNT_VMAX_OVERRIDE is not None else int(observed_global_max)
count_vmax = max(count_vmax, 1)

print("Observed max count:", observed_global_max)
print("Display count vmax:", count_vmax)


def make_count_cmap_and_norm(vmax):
    """Create a discrete Matplotlib colormap for absolute mouse-count values."""
    vmax = int(max(vmax, 1))
    base = plt.get_cmap(COUNT_CMAP, vmax)
    rgba = []

    for count in range(1, vmax + 1):
        if int(count) in COUNT_COLORS:
            rgba.append(mcolors.to_rgba(COUNT_COLORS[int(count)]))
        else:
            rgba.append(base(count - 1))

    cmap = ListedColormap(rgba, name="lesion_count_discrete")
    boundaries = np.arange(0.5, vmax + 1.5, 1.0)
    norm = BoundaryNorm(boundaries, cmap.N, clip=True)
    return cmap, norm, boundaries


count_cmap, count_norm, count_boundaries = make_count_cmap_and_norm(count_vmax)


## 10. Final `group × exp_day` figure

The bregma dot, atlas background, ROI contours, and optional available-cortex shading are display overlays. Count maps are saved separately from the figure.


In [ ]:
extent_mask = display_mask.copy()

if INCLUDE_LESION_IN_PLOT_EXTENT:
    for arr in count_maps.values():
        extent_mask |= np.asarray(arr) > 0

if INCLUDE_GROUP_WINDOW_IN_PLOT_EXTENT:
    for _, gmask in group_window_masks.items():
        if gmask is not None:
            extent_mask |= np.asarray(gmask, dtype=bool)

# Keep the display-only bregma point visible when plotting a cropped atlas.
if SHOW_BREGMA:
    extent_mask[BREGMA_Y, BREGMA_X] = True

plot_extent = crop_from_mask(extent_mask, pad=PLOT_PAD_PX)

n_rows = len(display_order)
n_cols = len(EXP_DAYS)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(FIGSIZE_PER_PANEL[0] * n_cols, FIGSIZE_PER_PANEL[1] * n_rows),
    squeeze=False,
)

last_im = None

for r, group_name in enumerate(display_order):
    row_label = localized_group_label(group_name)
    gmask = group_window_masks.get(str(group_name), None)

    for c, day in enumerate(EXP_DAYS):
        ax = axes[r, c]
        neutral_atlas(ax)

        if SHOW_GROUP_WINDOW and gmask is not None:
            draw_group_window_overlay(ax, gmask)

        count_map = count_maps[(str(group_name), str(day))]
        masked_count = np.ma.masked_where(count_map <= 0, count_map)

        last_im = ax.imshow(
            masked_count,
            cmap=count_cmap,
            norm=count_norm,
            interpolation="nearest",
            alpha=COUNT_ALPHA,
            zorder=8,
        )

        if SHOW_COUNT_BOUNDARY and np.any(count_map > 0):
            boundary = find_boundaries((count_map > 0).astype(np.uint8), mode="outer")
            draw_boundary(
                ax,
                boundary,
                color=COUNT_BOUNDARY_COLOR,
                lw=COUNT_BOUNDARY_LW,
                alpha=0.9,
                zorder=9,
            )

        draw_analysis_roi_boundaries(ax)
        draw_bregma_marker(ax)

        hit = group_day_summary[
            (group_day_summary["analysis_group"].astype(str) == str(group_name))
            & (group_day_summary["exp_day"].astype(str) == str(day))
        ]
        n_mice = int(hit.iloc[0]["n_mice"]) if not hit.empty else 0

        if r == 0:
            ax.set_title(
                localized_exp_day_label(day),
                fontsize=COLUMN_LABEL_FONT_SIZE,
                pad=8,
            )

        ax.text(
            0.02,
            0.02,
            f"n={n_mice}",
            transform=ax.transAxes,
            fontsize=N_TEXT_FONT_SIZE,
            ha="left",
            va="bottom",
            color="black",
            zorder=20,
            bbox=dict(facecolor="white", alpha=0.55, edgecolor="none", pad=1.5),
        )

        style_axes(ax, plot_extent)


fig.subplots_adjust(
    left=MARGIN_LEFT,
    right=MARGIN_RIGHT,
    bottom=MARGIN_BOTTOM,
    top=MARGIN_TOP,
    wspace=GRID_WSPACE,
    hspace=GRID_HSPACE,
)

# Row labels are positioned after layout, for any number of YAML groups.
for r, group_name in enumerate(display_order):
    bbox = axes[r, 0].get_position()
    fig.text(
        ROW_LABEL_X_FIG,
        0.5 * (bbox.y0 + bbox.y1),
        localized_group_label(group_name),
        rotation=ROW_LABEL_ROTATION,
        va="center",
        ha="center",
        fontsize=ROW_LABEL_FONT_SIZE,
    )

cax = fig.add_axes(COLORBAR_RECT)
if last_im is not None:
    cbar = fig.colorbar(
        last_im,
        cax=cax,
        ticks=np.arange(1, count_vmax + 1),
    )
    if str(LABEL_LANGUAGE).lower() == "ru":
        cbar.set_label("Число мышей", fontsize=COLORBAR_LABEL_FONT_SIZE)
    else:
        cbar.set_label("Number of mice", fontsize=COLORBAR_LABEL_FONT_SIZE)
    cbar.ax.tick_params(labelsize=COLORBAR_TICK_FONT_SIZE)

if SUPTITLE:
    fig.suptitle(SUPTITLE, fontsize=SUPTITLE_FONT_SIZE)

main_png = figures_root / "lesion_overlap_group_by_exp_day_count_windowed.png"
main_tiff = figures_root / "lesion_overlap_group_by_exp_day_count_windowed.tiff"

if SAVE_PNG:
    fig.savefig(main_png, dpi=DPI, 
                #bbox_inches="tight", 
                facecolor="white")
if SAVE_TIFF:
    try:
        fig.savefig(
            main_tiff,
            dpi=DPI,
            #bbox_inches="tight",
            facecolor="white",
            pil_kwargs={"compression": "tiff_lzw"},
        )
    except TypeError:
        fig.savefig(main_tiff, dpi=DPI, 
                    #bbox_inches="tight", 
                    facecolor="white")

plt.show()

print("Saved:")
if SAVE_PNG:
    print(" ", main_png)
if SAVE_TIFF:
    print(" ", main_tiff)

## 11. Optional: save separate panels

Separate panels are saved when `SAVE_SEPARATE_PANELS = True` (currently enabled). The main figure has independent `SAVE_PNG` and `SAVE_TIFF` switches.


In [ ]:
if SAVE_SEPARATE_PANELS:
    for group_name in display_order:
        gmask = group_window_masks.get(str(group_name), None)
        for day in EXP_DAYS:
            fig, ax = plt.subplots(figsize=FIGSIZE_PER_PANEL)
            neutral_atlas(ax)

            if SHOW_GROUP_WINDOW and gmask is not None:
                draw_group_window_overlay(ax, gmask)

            count_map = count_maps[(str(group_name), str(day))]
            masked_count = np.ma.masked_where(count_map <= 0, count_map)

            im = ax.imshow(
                masked_count,
                cmap=count_cmap,
                norm=count_norm,
                interpolation="nearest",
                alpha=COUNT_ALPHA,
                zorder=8,
            )

            draw_analysis_roi_boundaries(ax)
            draw_bregma_marker(ax)

            hit = group_day_summary[
                (group_day_summary["analysis_group"].astype(str) == str(group_name))
                & (group_day_summary["exp_day"].astype(str) == str(day))
            ]
            n_mice = int(hit.iloc[0]["n_mice"]) if not hit.empty else 0

            title = f"{localized_group_label(group_name)} | {localized_exp_day_label(day)} | n={n_mice}"
            ax.set_title(title, fontsize=TITLE_FONT_SIZE)
            style_axes(ax, plot_extent)

            cb = fig.colorbar(
                im,
                ax=ax,
                fraction=0.035,
                pad=0.02,
                ticks=np.arange(1, count_vmax + 1),
            )
            if str(LABEL_LANGUAGE).lower() == "ru":
                cb.set_label("Число мышей", fontsize=COLORBAR_LABEL_FONT_SIZE)
            else:
                cb.set_label("Number of mice", fontsize=COLORBAR_LABEL_FONT_SIZE)
            cb.ax.tick_params(labelsize=COLORBAR_TICK_FONT_SIZE)

            fig.tight_layout()

            base = f"{group_name}__{day}__lesion_count_windowed"
            if SAVE_PNG:
                fig.savefig(figures_root / f"{base}.png", dpi=DPI, bbox_inches="tight", facecolor="white")
            if SAVE_TIFF:
                tiff_path = figures_root / f"{base}.tiff"
                try:
                    fig.savefig(
                        tiff_path,
                        dpi=DPI,
                        bbox_inches="tight",
                        facecolor="white",
                        pil_kwargs={"compression": "tiff_lzw"},
                    )
                except TypeError:
                    fig.savefig(tiff_path, dpi=DPI, bbox_inches="tight", facecolor="white")

            plt.close(fig)

    print("Separate panels saved to:", figures_root)


## 12. Representative mean-sized lesion scheme

This section creates **one canonical lesion spot with an approximately mean lesion area**.

Procedure:
1. Collect all `mouse × exp_day` union masks.
2. Compute the pooled absolute count map.
3. Calculate `mean_area_px = mean(area(mask_i))`.
4. Select `K = round(mean_area_px)` pixels with the highest observed frequencies, capped by the number of pixels with a positive count.
5. Construct one binary representative mask.

This represents the **mean lesion size**, not the union of all lesions and not a fraction map. Equal-frequency ties are resolved by the stable ordering of the current code.


In [ ]:
all_mouse_day_items = sorted(mouse_day_masks.items(), key=lambda x: (x[0][0], x[0][1], x[0][2]))

if all_mouse_day_items:
    all_masks_stack = np.stack([m.astype(np.uint16) for _, m in all_mouse_day_items], axis=0)
    pooled_count_all = all_masks_stack.sum(axis=0).astype(np.uint16)
    areas = np.array([int(np.sum(m)) for _, m in all_mouse_day_items], dtype=float)
    mean_area_px = int(np.round(np.mean(areas)))
else:
    pooled_count_all = np.zeros(canonical_shape, dtype=np.uint16)
    areas = np.array([], dtype=float)
    mean_area_px = 0

flat = pooled_count_all.ravel()
positive_idx = np.flatnonzero(flat > 0)

representative_mean_mask = np.zeros(canonical_shape, dtype=bool)
selected_top_k = 0

if positive_idx.size > 0 and mean_area_px > 0:
    # Descending by frequency, stable ordering.
    order = positive_idx[np.argsort(-flat[positive_idx], kind="stable")]
    selected_top_k = int(min(mean_area_px, order.size))
    chosen = order[:selected_top_k]
    representative_mean_mask.ravel()[chosen] = True

representative_mean_mask_path = output_root / "representative_mean_lesion_mask_canonical.npy"
pooled_count_all_path = output_root / "all_mouse_day_pooled_count_canonical.npy"

np.save(representative_mean_mask_path, representative_mean_mask.astype(np.uint8))
np.save(pooled_count_all_path, pooled_count_all.astype(np.uint16))

mean_scheme_summary = pd.DataFrame(
    [{
        "n_mouse_day_masks": int(len(all_mouse_day_items)),
        "mean_lesion_area_px": int(mean_area_px),
        "positive_pixels_in_pooled_count": int(positive_idx.size),
        "selected_top_k_pixels": int(selected_top_k),
        "representative_mean_mask_area_px": int(np.sum(representative_mean_mask)),
        "representative_mean_mask_path": str(representative_mean_mask_path),
        "pooled_count_all_path": str(pooled_count_all_path),
    }]
)
mean_scheme_summary.to_csv(tables_root / "representative_mean_lesion_summary.csv", index=False)
display(mean_scheme_summary)

all_groups_window_union = np.zeros(canonical_shape, dtype=bool)
for gname in display_order:
    gmask = group_window_masks.get(str(gname), None)
    if gmask is not None:
        all_groups_window_union |= np.asarray(gmask, dtype=bool)


In [ ]:
if SHOW_MEAN_LESION_SCHEME:
    mean_extent_mask = display_mask.copy()
    mean_extent_mask |= representative_mean_mask

    if SHOW_MEAN_SCHEME_WINDOW and np.any(all_groups_window_union):
        mean_extent_mask |= all_groups_window_union

    # Preserve the bregma marker within the displayed crop only.
    if SHOW_BREGMA:
        mean_extent_mask[BREGMA_Y, BREGMA_X] = True

    mean_plot_extent = crop_from_mask(mean_extent_mask, pad=PLOT_PAD_PX)

    fig, ax = plt.subplots(figsize=MEAN_SCHEME_FIGSIZE)

    neutral_atlas(ax)

    if SHOW_MEAN_SCHEME_WINDOW and np.any(all_groups_window_union):
        draw_group_window_overlay(ax, all_groups_window_union)

    if np.any(representative_mean_mask):
        draw_fill(
            ax,
            representative_mean_mask,
            color=MEAN_LESION_FILL_COLOR,
            alpha=MEAN_LESION_FILL_ALPHA,
            zorder=8,
        )
        rep_boundary = find_boundaries(representative_mean_mask.astype(np.uint8), mode="outer")
        draw_boundary(
            ax,
            rep_boundary,
            color=MEAN_LESION_OUTLINE_COLOR,
            lw=MEAN_LESION_OUTLINE_LW,
            alpha=MEAN_LESION_OUTLINE_ALPHA,
            zorder=10,
        )

    draw_analysis_roi_boundaries(ax)
    draw_bregma_marker(ax)
    style_axes(ax, mean_plot_extent)

    if MEAN_SCHEME_TITLE:
        ax.set_title(MEAN_SCHEME_TITLE, fontsize=TITLE_FONT_SIZE)

    if str(LABEL_LANGUAGE).lower() == "ru":
        stat_text = (
            f"n={len(all_mouse_day_items)} masks\n"
            f"средняя площадь = {mean_area_px} px"
        )
    else:
        stat_text = (
            f"n={len(all_mouse_day_items)} masks\n"
            f"mean area = {mean_area_px} px"
        )

#    ax.text(
#        0.02,
#        0.02,
 #       stat_text,
#        transform=ax.transAxes,
#        fontsize=MEAN_SCHEME_TEXT_FONT_SIZE,
#        ha="left",
#        va="bottom",
#        color="black",
#        zorder=20,
#        bbox=dict(facecolor="white", alpha=0.60, edgecolor="none", pad=1.6),
 #   )

    mean_png = figures_root / "representative_mean_lesion_scheme.png"
    mean_tiff = figures_root / "representative_mean_lesion_scheme.tiff"

    if SAVE_PNG:
        fig.savefig(mean_png, dpi=DPI, bbox_inches="tight", facecolor="white")
    if SAVE_TIFF:
        try:
            fig.savefig(
                mean_tiff,
                dpi=DPI,
                bbox_inches="tight",
                facecolor="white",
                pil_kwargs={"compression": "tiff_lzw"},
            )
        except TypeError:
            fig.savefig(mean_tiff, dpi=DPI, bbox_inches="tight", facecolor="white")

    plt.show()

    print("Saved representative mean lesion scheme:")
    if SAVE_PNG:
        print(" ", mean_png)
    if SAVE_TIFF:
        print(" ", mean_tiff)


## 13. Outputs

The notebook displays the group window summary, group-by-day lesion counts, representative mean-lesion summary, output file locations, and inclusion/exclusion QC tables.


In [ ]:
print("=== GROUP WINDOW SUMMARY ===")
display(group_window_summary)

print("\n=== GROUP × DAY SUMMARY ===")
display(group_day_summary)

print("\n=== REPRESENTATIVE MEAN LESION SUMMARY ===")
display(mean_scheme_summary)

print("\nMain figure:")
print(" ", figures_root / "lesion_overlap_group_by_exp_day_count_windowed.png")

print("\nRepresentative mean lesion figure:")
print(" ", figures_root / "representative_mean_lesion_scheme.png")

print("\nImportant outputs:")
print(" ", tables_root / "lesion_canonical_projection_qc.csv")
print(" ", tables_root / "lesion_mouse_day_canonical_qc.csv")
print(" ", tables_root / "group_available_window_summary.csv")
print(" ", tables_root / "lesion_group_day_canonical_summary.csv")
print(" ", tables_root / "representative_mean_lesion_summary.csv")
print(" ", output_root / "representative_mean_lesion_mask_canonical.npy")
print(" ", output_root / "all_mouse_day_pooled_count_canonical.npy")

print("\nSelection and exclusions QC:")
print(" ", tables_root / "lesion_selection_audit.csv")
print(" ", tables_root / "analysis_exclusion_rule_audit.csv")
print(" ", metadata_csv)